# Azure Document Intelligence PDF Parser

Parse PDF files using Azure's prebuilt-document model to extract structured content.

**Note:** Credentials are loaded from `parser.env` file in the project root.

## 1. Import Required Libraries

%pip install azure-ai-documentintelligence python-dotenv

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential
import json

## 2. Set Azure Credentials

In [3]:
# Load credentials from parser.env file
load_dotenv("parser.env")

endpoint = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT")
key = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_KEY")

if not endpoint or not key:
    print("⚠️  Azure credentials not found in parser.env!")
    print("Make sure parser.env contains:")
    print("  AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT=\"https://<resource>.cognitiveservices.azure.com/\"")
    print("  AZURE_DOCUMENT_INTELLIGENCE_KEY=\"your-api-key\"")
else:
    print(f"✓ Endpoint: {endpoint}")
    print(f"✓ Key configured: {key[:5]}...")

✓ Endpoint: https://med-parser.cognitiveservices.azure.com/
✓ Key configured: 8Mes7...


## 3. Initialize Azure Document Intelligence Client

In [4]:
# Initialize client
client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))
print("✓ Client initialized")

✓ Client initialized


## 4. Parse PDF with prebuilt-document Model

In [6]:
# Path to PDF file
pdf_path = "Claims Appeal Forms/AETNA_Form1.pdf"

# Verify file exists
if not Path(pdf_path).exists():
    print(f"✗ File not found: {pdf_path}")
else:
    print(f"✓ File found: {pdf_path}")
    print(f"  Size: {Path(pdf_path).stat().st_size / 1024:.1f} KB")

✓ File found: Claims Appeal Forms/AETNA_Form1.pdf
  Size: 184.9 KB


In [12]:
AZURE_MODEL = "prebuilt-layout"

In [20]:
with open(pdf_path, 'rb') as pdf_file:
	poller = client.begin_analyze_document(model_id = AZURE_MODEL, 
                                    features="keyValuePairs",
									document = pdf_file)

result = poller.result()

TypeError: DocumentIntelligenceClientOperationsMixin.begin_analyze_document() missing 1 required positional argument: 'body'

In [ ]:
lines = []

for page in result.pages:
	for line in page.lines:
		text = line.content.strip()

		if len(text) < 2:
			continue

		lines.append(text)



['New Jersey Department of Banking and Insurance', 'Health Care Provider Application to Appeal a Claims Determination', 'aetna®', 'Aetna - Provider Resolution Team', 'P.O. Box 14020', 'Lexington, KY 40512', 'Or fax to: (859) 455-8650', 'You have the right to appeal Our1 claims determination(s) on claims you submitted to Us. You also have the right to', 'appeal an apparent lack of activity on a claim you submitted.', 'DO NOT submit a Health Care Provider Application to Appeal a Claims Determination IF:', '> Our determination indicates that We concluded the health care services for which the claim was submitted were', 'not medically necessary, were experimental or investigational, were cosmetic rather than medically necessary or', 'dental rather than medical. INSTEAD, you may submit a request for a Stage 1 UM Appeal Review to appeal such', 'determinations. For more information, contact the Aetna Provider Service Center.', '> Our determination indicates that We considered the person to wh

New Jersey Department of Banking and Insurance
Health Care Provider Application to Appeal a Claims Determination
aetna®
Aetna - Provider Resolution Team
P.O. Box 14020
Lexington, KY 40512
Or fax to: (859) 455-8650
You have the right to appeal Our1 claims determination(s) on claims you submitted to Us. You also have the right to
appeal an apparent lack of activity on a claim you submitted.
DO NOT submit a Health Care Provider Application to Appeal a Claims Determination IF:
> Our determination indicates that We concluded the health care services for which the claim was submitted were
not medically necessary, were experimental or investigational, were cosmetic rather than medically necessary or
dental rather than medical. INSTEAD, you may submit a request for a Stage 1 UM Appeal Review to appeal such
determinations. For more information, contact the Aetna Provider Service Center.
> Our determination indicates that We considered the person to whom health care services for which the claim 

## 8. Agent 1 — Form Schema Extraction (OpenAI)

Use an LLM to convert the raw OCR text from Azure into a structured JSON schema
describing every fillable field on the form. This schema is the contract that
Agent 2 (filler) and Agent 3 (PDF writer) will consume.

In [ ]:
%pip install openai

In [ ]:
# Initialize OpenAI client (key already loaded from parser.env in section 2)
from openai import OpenAI

openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    print("⚠️  OPENAI_API_KEY not found in parser.env!")
    print("Add this line to parser.env:")
    print('  OPENAI_API_KEY="sk-..."')
else:
    print(f"✓ OpenAI key configured: {openai_key[:7]}...")

oai_client = OpenAI(api_key=openai_key)
LLM_MODEL = "gpt-5-mini"
print(f"✓ OpenAI client initialized")
print(f"  Model: {LLM_MODEL}")

### 8.1 Prepare the input for Agent 1

For this first pass, we feed the LLM the full OCR text (`result.content`).
We'll layer in spatial info (bounding polygons) once we see whether the
schema looks correct.

In [ ]:
form_text = result.content

print(f"Form text length: {len(form_text)} characters")
print(f"Approx tokens (rough estimate, chars/4): {len(form_text) // 4}")
print(f"\nFirst 400 chars:")
print("-" * 60)
print(form_text[:400])
print("-" * 60)

### 8.2 Define the output JSON schema with Pydantic

OpenAI's structured-output API validates the LLM's response against this
Pydantic model, so we get guaranteed-valid JSON back (no parse errors).

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional


class FormField(BaseModel):
    field_id: str = Field(
        description="A stable, unique snake_case identifier derived from the label, e.g. 'provider_name'."
    )
    section: str = Field(
        description="The section heading this field belongs to, e.g. 'A. Provider Information'."
    )
    label: str = Field(
        description="The exact human-readable label as printed on the form (without trailing colon)."
    )
    field_type: Literal[
        "text", "date", "phone", "email", "tin_npi_or_ssn", "currency",
        "number", "address", "long_text", "signature", "checkbox", "radio_group"
    ] = Field(description="The semantic type of the field.")
    options: Optional[list[str]] = Field(
        description="For checkbox or radio_group fields, the list of available options (e.g. ['Yes', 'No', 'NA']). Null otherwise."
    )
    format_hint: Optional[str] = Field(
        description="Format hint if applicable, e.g. 'MM/DD/YYYY' or 'XX-XXXXXXX'. Null if none."
    )
    required: bool = Field(
        description="Whether the field appears required on the form (look for 'Required' labels nearby)."
    )


class FormSchema(BaseModel):
    form_title: str = Field(description="The official form title.")
    issuer: str = Field(description="The insurance carrier / issuer, e.g. 'Aetna', 'UHC', 'Anthem'.")
    sections: list[str] = Field(description="Ordered list of all section headings on the form.")
    fields: list[FormField] = Field(description="All fillable fields on the form, in reading order.")


print("✓ Pydantic schema defined")
print(f"  FormField fields: {list(FormField.model_fields.keys())}")
print(f"  FormSchema fields: {list(FormSchema.model_fields.keys())}")

### 8.3 Call Agent 1 to extract the schema

In [ ]:
SYSTEM_PROMPT = """You are an expert at analyzing healthcare insurance claims forms.

You will receive the OCR-extracted text of a healthcare claims appeal form. Your job is
to identify every fillable field on the form and produce a structured JSON schema
describing them.

Rules:
1. IGNORE instructional / body text (paragraphs explaining when to submit the form,
   eligibility text, legal disclaimers, page footers, etc.). Only capture actual
   fillable fields.
2. Identify section headers (e.g. "A. Provider Information", "B. Patient Information",
   "C. Claim Information", "D. Reason for Appeal").
3. For each field, infer the most specific semantic type from the allowed list.
4. For Yes/No or Yes/No/NA choice groups, use field_type="radio_group" with options.
5. For grouped checkboxes where multiple may be selected (e.g. "Check all that apply"),
   use field_type="checkbox" with options.
6. Use stable snake_case field_ids derived from the label (e.g. "provider_name",
   "patient_ins_id", "claim_number").
7. Mark required=true only when the form explicitly indicates it (e.g. "(Required)").
8. Be EXHAUSTIVE. Capture every numbered or labeled field across all pages.
"""

user_msg = f"OCR-extracted text of the form:\n\n{form_text}"

print(f"Calling {LLM_MODEL}...")
response = oai_client.chat.completions.parse(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ],
    response_format=FormSchema,
)

form_schema = response.choices[0].message.parsed
print("✓ Schema extracted")
print(f"\nToken usage:")
print(f"  prompt:     {response.usage.prompt_tokens}")
print(f"  completion: {response.usage.completion_tokens}")
print(f"  total:      {response.usage.total_tokens}")

### 8.4 Inspect the extracted schema

In [ ]:
from collections import Counter

print(f"Form: {form_schema.form_title}")
print(f"Issuer: {form_schema.issuer}")
print(f"\nSections ({len(form_schema.sections)}):")
for s in form_schema.sections:
    print(f"  - {s}")

print(f"\nTotal fields extracted: {len(form_schema.fields)}")

type_counts = Counter(f.field_type for f in form_schema.fields)
print(f"\nField breakdown by type:")
for t, c in type_counts.most_common():
    print(f"  {t:20s} {c}")

print(f"\nFields per section:")
section_counts = Counter(f.section for f in form_schema.fields)
for s, c in section_counts.items():
    print(f"  {s}: {c}")

In [ ]:
import json

print("=" * 70)
print("FULL EXTRACTED SCHEMA (JSON)")
print("=" * 70)
print(json.dumps(form_schema.model_dump(), indent=2))

## 5. Structure Results in doc_context Variable

In [ ]:
# Create doc_context variable with parsed results
doc_context = {
    "status": "success",
    "model": "prebuilt-document",
    "file": Path(pdf_path).name,
    "content": result.content,
    "pages": result.pages,
    "tables": result.tables if hasattr(result, 'tables') else [],
    "paragraphs": result.paragraphs if hasattr(result, 'paragraphs') else [],
    "key_value_pairs": result.key_value_pairs if hasattr(result, 'key_value_pairs') else [],
    "raw_result": result,
}

print("✓ doc_context created")
print(f"\nSummary:")
print(f"  File: {doc_context['file']}")
print(f"  Model: {doc_context['model']}")
print(f"  Content length: {len(doc_context['content'])} characters")
print(f"  Pages: {len(doc_context['pages'])}")
print(f"  Tables: {len(doc_context['tables'])}")
print(f"  Paragraphs: {len(doc_context['paragraphs'])}")
print(f"  Key-value pairs: {len(doc_context['key_value_pairs'])}")

## 6. Explore Extracted Content

In [ ]:
# Display first 500 characters of extracted content
print("Extracted Content (first 500 chars):")
print("=" * 50)
print(doc_context['content'][:500])
print("...")

In [ ]:
# Display page information
if doc_context['pages']:
    print(f"\nPage Information:")
    print("=" * 50)
    for i, page in enumerate(doc_context['pages'][:3], 1):  # Show first 3 pages
        print(f"\nPage {i}:")
        print(f"  Height: {page.height if hasattr(page, 'height') else 'N/A'}")
        print(f"  Width: {page.width if hasattr(page, 'width') else 'N/A'}")
        print(f"  Unit: {page.unit if hasattr(page, 'unit') else 'N/A'}")
        if hasattr(page, 'lines'):
            print(f"  Lines: {len(page.lines)}")

In [ ]:
# Display tables if any
if doc_context['tables']:
    print(f"\nTables Found: {len(doc_context['tables'])}")
    print("=" * 50)
    for i, table in enumerate(doc_context['tables'][:2], 1):  # Show first 2 tables
        print(f"\nTable {i}:")
        print(f"  Rows: {table.row_count if hasattr(table, 'row_count') else 'N/A'}")
        print(f"  Columns: {table.column_count if hasattr(table, 'column_count') else 'N/A'}")
        if hasattr(table, 'cells'):
            print(f"  Cells: {len(table.cells)}")
else:
    print("\nNo tables found in document")

## 7. Access doc_context Variables Directly

In [ ]:
# Access specific parts of doc_context

# Get full text content
text_content = doc_context['content']
print(f"Full text length: {len(text_content)} characters")

# Get pages
pages = doc_context['pages']
print(f"Total pages: {len(pages)}")

# Get tables
tables = doc_context['tables']
print(f"Total tables: {len(tables)}")

# Get key-value pairs (form fields)
kvp = doc_context['key_value_pairs']
print(f"Total key-value pairs: {len(kvp)}")

# Get raw result for detailed inspection
raw = doc_context['raw_result']
print(f"Raw result type: {type(raw)}")